In [ ]:
import os
os.chdir(path_to_wd)
import sys

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pycats
from pandas.api.types import CategoricalDtype
import scvelo as scv

scvi.settings.seed = 1234
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

In [ ]:
tmp_subset = 'BCG'

fig_dir = f"Reproducibility/Results/Plots/{tmp_subset}/"
os.makedirs(fig_dir, exist_ok = True)

adata = adata_full[adata_full.obs["sample"].isin(["BC_011","BC_016","BC_023","BC_033","BC_037",
                   "BC_027","BC_032","BC_039","BC_040","BC_043","BC_044","BC_047","BC_048"])&
                   adata_full.obs['lineage'].isin(['CD4_T','CD8_T_NK_ILC','B','Myeloid'])].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
pre = ["BC_011","BC_016","BC_023","BC_033","BC_037"]
post = ["BC_027","BC_032","BC_039","BC_040","BC_043","BC_044","BC_047","BC_048"]

BCG = {
    'pre': pre,
    'post': post
}

adata.obs['BCG'] = adata.obs["sample"].astype('category')
adata.obs['BCG'] = pycats.cat_collapse(adata.obs["BCG"], BCG)

cat_type = CategoricalDtype(categories=["pre","post"], ordered=True)
adata.obs['BCG'] = adata.obs['BCG'].astype(cat_type)

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["CD4_Tconv","Treg",'CD8_T',"NK_ILC","B",
                                        "Plasma","Mono_Mac",'DC','Mast'], ordered=True)
adata.obs['coarse_celltype2'] = adata.obs['coarse_celltype'].astype(cat_type)
adata.obs.loc[adata.obs['celltype'] == 'Plasma', 'coarse_celltype2'] = 'Plasma'

In [ ]:
# Fig.S10A
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=["BCG"],
    frameon=False,
    ncols=1,
    palette=["#66c2a5","#fc8d62"],   
    legend_fontsize=7,
    show = False
)
plt.savefig("Reproducibility/Results/Plots/BCG/FigureS10A_time.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S10A
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=["coarse_celltype2"],
    frameon=False,
    ncols=1,
    legend_fontsize=7,
    show = False
)
plt.savefig("Reproducibility/Results/Plots/BCG/FigureS10A_coarse_celltype.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S10G
adata = sc.read("Reproducibility/Data/adata_concat_w_pancancer_CD4_T.h5ad")
adata = adata[adata.obs['cancerType'].isin(['CHOL'])==False]

In [ ]:
tmp_path = "Reproducibility/Data/embeddings/UC_DOGMA_pancancer_CD4_T_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

In [ ]:
post_BCG = ["BC_027","BC_032","BC_039","BC_040","BC_043","BC_044","BC_047","BC_048"]

adata.obs['cancerType2'] = adata.obs['cancerType']
adata.obs['cancerType2'] = np.where(
    adata.obs['patient'].isin(post_BCG),              # Condition
    'BCG',                                            # Value if True
    adata.obs['cancerType']                           # Value if False
)

In [ ]:
pancancer = ["BC","BCL","ESCA","FTC","MM","OV","PACA",'RC','THCA','UCEC']

adata.obs['cancerType2'] = np.where(
    adata.obs['cancerType2'].isin(pancancer),          # Condition
    'pan-cancer',                                      # Value if True
    adata.obs['cancerType2']                           # Value if False
)

sc.tl.embedding_density(adata, basis='umap', groupby='cancerType2')

In [ ]:
colors = ["#e8e9eb", "#f3f4db", "#ffffcc", "#ffcc66", "#ff3300", "#990000"]
custom_cmap = LinearSegmentedColormap.from_list("soft_heatmap", colors)

sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.embedding_density(
    adata, 
    basis='umap', 
    key='umap_density_cancerType2', 
    color_map=custom_cmap,
    bg_dotsize=2,
    fg_dotsize=5,
    group="all",
    show=False
)

plt.savefig("Reproducibility/Results/Plots/BCG/Figure5G.pdf", bbox_inches='tight')
plt.close()

In [ ]:
scv.set_figure_params(vector_friendly=True, dpi_save=100)
scv.pl.scatter(adata, 
               dpi = 100,
               groups=['CD4_T_CD26'], 
               color='meta.cluster', 
               frameon=False,
               ncols=5,
               size = 1,
               linewidth = 0,
               outline_width = 0,
               figsize = (3,3),
               show = False,
               )

plt.savefig("Reproducibility/Results/Plots/BCG/Figure5G_CD4_T_CD26.pdf", bbox_inches='tight')
plt.close()